# Chuẩn bị dữ liệu StarDist
Notebook dùng trên máy cá nhân hoặc Google Colab. Gói dữ liệu đã được xử lý và kiểm tra; có thể sử dụng trực tiếp hoặc tái tạo từ dữ liệu gốc.

**Colab:** giải nén gói ZIP vào `/content`, mở notebook và đặt `PROJECT` trỏ tới thư mục chứa `dataset_manifest.csv`. Quy trình chuẩn bị dữ liệu không cần GPU.

In [ ]:
%pip -q install numpy Pillow pandas


In [ ]:
from pathlib import Path
import json, sys, subprocess
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
PROJECT = Path.cwd()  # Colab: Path('/content/StarDist_Dataset')
assert (PROJECT / 'dataset_manifest.csv').exists(), 'Sửa PROJECT thành thư mục gói dữ liệu'


## Tái tạo từ dữ liệu gốc (tùy chọn)
Mỗi PNG mask tương ứng một nhân. Script kiểm tra mask rỗng, sai kích thước, sai giá trị và chồng lấn; sau đó ghi TIFF uint16, nền 0 và ID 1..N. Chia theo nhóm ảnh trùng pixel, phân tầng theo diện mạo và số nhân, seed 42. Giữ nguyên ảnh PNG, không resize.

Để chạy lại: tải `stage1_train.zip` lên Colab và giải nén; sửa `RAW`, `NEW_OUTPUT`, bật `RUN_PREPARATION`. Đầu ra phải là thư mục mới để tránh trộn kết quả.

In [ ]:
RUN_PREPARATION = False
RAW = Path('/content/stage1_train')
NEW_OUTPUT = Path('/content/StarDist_Project_rebuilt')
if RUN_PREPARATION:
    subprocess.run([sys.executable, str(PROJECT / 'prepare_dataset.py'), '--raw', str(RAW), '--out', str(NEW_OUTPUT)], check=True)


## Thống kê và danh sách dữ liệu pretrained
Thí nghiệm sử dụng pretrained, không train/fine-tune. `dataset_manifest.csv` chứa toàn bộ dữ liệu hợp lệ. `experiments/pretrained_fluo/manifest.csv` dùng 20 ảnh validation và 20 ảnh test từ tập khởi đầu xám nền tối; giữ nguyên split. Phần train được giữ dự phòng. Mô hình đề xuất `2D_versatile_fluo` cần xác nhận miền ảnh bằng quan sát; cả mô hình này và `2D_paper_dsb2018` đã học một phần DSB2018 nên test cục bộ chưa phải kiểm thử tổng quát hóa độc lập. Đọc `HUONG_DAN_PRETRAINED.md`.

In [ ]:
df = pd.read_csv(PROJECT / 'dataset_manifest.csv', keep_default_na=False)
display(df.groupby('split').agg(images=('source_id','count'), nuclei=('num_instances','sum')))
display(pd.crosstab(df['appearance_group'], df['split']))
primary = pd.read_csv(PROJECT / 'experiments/pretrained_fluo/manifest.csv', keep_default_na=False)
display(primary.groupby('split').agg(images=('source_id','count'), nuclei=('num_instances','sum')))


## Kiểm tra ảnh, nhãn và chuẩn hóa
Màu trong ảnh nhãn chỉ để hiển thị. TIFF lưu ID số nguyên. Ảnh xám nền tối có thêm `.npy` float32 theo công thức clip((I−P1)/(P99.8−P1),0,1). Không làm mượt hay chuẩn hóa nhãn.

In [ ]:
display(Image.open(PROJECT / 'results/quality_control.png'))
display(Image.open(PROJECT / 'results/normalization_example.png'))
r = primary[primary['split'] == 'test'].iloc[0]
image = np.asarray(Image.open(PROJECT / r.image_path))
labels = np.asarray(Image.open(PROJECT / r.mask_path))
normalized = np.load(PROJECT / r.normalized_path)
print('Original:', image.shape, image.dtype)
print('GT:', labels.shape, labels.dtype, 'instances:', len(np.unique(labels[labels > 0])))
print('Normalized:', normalized.shape, normalized.dtype, normalized.min(), normalized.max())


## Kiểm tra gói dữ liệu
Kiểm tra kích thước, ID, hash ảnh gốc và phân chia; bước tái tạo ở trên kiểm tra trực tiếp từng mask nguồn.

In [ ]:
sys.path.insert(0, str(PROJECT))
from prepare_dataset import verify
records = df.to_dict('records')
print(json.dumps(verify(PROJECT, records), indent=2))


## Sử dụng và xuất dữ liệu
Các thuật toán sử dụng cùng manifest và danh sách test. Chọn tham số trên validation, đánh giá cuối cùng trên test. Đọc `README.md`, `BAO_CAO_DU_LIEU.md` và `quality_report.json`. Random split không bảo đảm độc lập về thí nghiệm/nguồn hoặc với dữ liệu huấn luyện của mô hình pretrained.

Để nén lại sau khi thay đổi: chạy ô sau. ZIP nằm bên ngoài thư mục PROJECT, tránh tự nén chính nó.

In [ ]:
REPACK = False
if REPACK:
    import shutil
    archive = shutil.make_archive(str(PROJECT.parent / (PROJECT.name + '_handover')), 'zip', PROJECT.parent, PROJECT.name)
    print(archive)
